# Loss Functions in PyTorch

For each loss function we:
1. **Implement from scratch** using basic tensor ops
2. **Verify numerically** against `torch.nn`
3. **Verify the gradient** via `.backward()` vs the analytical gradient

By the end you will understand why PyTorch's built-in losses use the specific formulations they do.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')

def check(name, a, b, atol=1e-5):
    ok = torch.allclose(a.float(), b.float(), atol=atol)
    status = '✓' if ok else '✗'
    print(f'{status} {name}: max_diff={torch.abs(a.float()-b.float()).max().item():.2e}')

## Section 1 — Regression Losses: L1, MSE, Huber, SmoothL1

In [ ]:
inp = torch.tensor([ 1.5, -0.5,  2.0,  0.0,  3.0])
tgt = torch.tensor([ 1.0,  1.0,  0.0, -1.0,  2.5])

# --- L1Loss (MAE) ---
def l1_scratch(x, y): return torch.abs(x - y).mean()
check('L1Loss', l1_scratch(inp, tgt), nn.L1Loss()(inp, tgt))

# Analytical gradient: sign(x-y)/N
x = inp.clone().requires_grad_(True)
nn.L1Loss()(x, tgt).backward()
analytical = torch.sign(inp - tgt) / inp.numel()
check('L1 gradient', x.grad, analytical)

# --- MSELoss ---
def mse_scratch(x, y): return ((x - y)**2).mean()
check('MSELoss', mse_scratch(inp, tgt), nn.MSELoss()(inp, tgt))

x = inp.clone().requires_grad_(True)
nn.MSELoss()(x, tgt).backward()
analytical = 2 * (inp - tgt) / inp.numel()
check('MSE gradient', x.grad, analytical)

# --- HuberLoss (delta=1) ---
def huber_scratch(x, y, delta=1.0):
    diff = torch.abs(x - y)
    return torch.where(diff <= delta,
                       0.5 * diff**2,
                       delta * (diff - 0.5 * delta)).mean()
check('HuberLoss', huber_scratch(inp, tgt), nn.HuberLoss(delta=1.0)(inp, tgt))

# --- SmoothL1Loss (beta=1) ---
def smooth_l1_scratch(x, y, beta=1.0):
    diff = torch.abs(x - y)
    return torch.where(diff < beta,
                       0.5 * diff**2 / beta,
                       diff - 0.5 * beta).mean()
check('SmoothL1Loss', smooth_l1_scratch(inp, tgt), nn.SmoothL1Loss(beta=1.0)(inp, tgt))

In [ ]:
# Visualise gradient shapes
errors = torch.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Functions
ax = axes[0]
ax.plot(errors, torch.abs(errors), label='L1', linewidth=2)
ax.plot(errors, errors**2, label='MSE', linewidth=2)
ax.plot(errors, torch.where(errors.abs()<=1, 0.5*errors**2, errors.abs()-0.5), label='Huber(δ=1)', linewidth=2)
ax.set_xlabel('error (x - y)'); ax.set_ylabel('loss'); ax.set_title('Loss vs Error')
ax.legend(); ax.set_ylim(-0.1, 5); ax.grid(True, alpha=0.3)

# Gradients
ax = axes[1]
ax.plot(errors, torch.sign(errors), label='L1 gradient (±1)', linewidth=2)
ax.plot(errors, 2*errors, label='MSE gradient (2e)', linewidth=2)
huber_grad = torch.where(errors.abs()<=1, errors, torch.sign(errors))
ax.plot(errors, huber_grad, label='Huber gradient', linewidth=2)
ax.set_xlabel('error (x - y)'); ax.set_ylabel('gradient'); ax.set_title('Gradient vs Error')
ax.legend(); ax.set_ylim(-3, 3); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 2 — Probabilistic Regression: PoissonNLL, GaussianNLL

Both losses are derived from maximum likelihood estimation (MLE). We negate the log-likelihood of the data under the assumed distribution.

In [ ]:
# PoissonNLLLoss
# Model predicts log(λ); target is observed count y
# P(y|λ) = λ^y * e^{-λ} / y!  →  -log P = λ - y*log(λ) = exp(x) - y*x (ignoring log(y!))
log_input = torch.tensor([0.5, 1.0, 1.5, 2.0])  # log(λ)
target_counts = torch.tensor([1.0, 2.0, 2.0, 3.0])

def poisson_nll_scratch(log_lam, y):
    return (torch.exp(log_lam) - y * log_lam).mean()

pytorch_pnll = nn.PoissonNLLLoss(log_input=True, reduction='mean')(log_input, target_counts)
check('PoissonNLL', poisson_nll_scratch(log_input, target_counts), pytorch_pnll)

print(f'\nPoissonNLL example:')
print(f'  log(λ) = {log_input.tolist()}')
print(f'  λ      = {torch.exp(log_input).tolist()}')
print(f'  y      = {target_counts.tolist()}')
print(f'  loss   = {pytorch_pnll.item():.4f}')

In [ ]:
# GaussianNLLLoss
# P(y|μ,σ²) = N(y; μ, σ²)  →  -log P = 0.5*(log(σ²) + (μ-y)²/σ²)
mu  = torch.tensor([1.0, 2.0, 3.0, 4.0])
var = torch.tensor([0.5, 1.0, 2.0, 0.1])  # must be positive
y   = torch.tensor([1.2, 1.8, 3.5, 4.1])

def gaussian_nll_scratch(mean, y, var):
    return (0.5 * (torch.log(var) + (mean - y)**2 / var)).mean()

pytorch_gnll = nn.GaussianNLLLoss()(mu, y, var)
check('GaussianNLL', gaussian_nll_scratch(mu, y, var), pytorch_gnll)

print('\nGaussianNLL per-sample decomposition:')
log_part = 0.5 * torch.log(var)
sq_part  = 0.5 * (mu - y)**2 / var
for i in range(4):
    print(f'  i={i}: 0.5*log(σ²)={log_part[i]:.3f}, (μ-y)²/2σ²={sq_part[i]:.3f}, total={log_part[i]+sq_part[i]:.3f}')

## Section 3 — Binary Classification: BCELoss, BCEWithLogitsLoss

Numerical stability: BCELoss requires probabilities and fails for large logits. BCEWithLogitsLoss uses the log-sum-exp trick.

In [ ]:
logits = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
probs  = torch.sigmoid(logits)
target = torch.tensor([0.0,  0.0, 1.0, 1.0, 1.0])

# BCELoss (requires probabilities)
def bce_scratch(p, y, eps=1e-7):
    p = torch.clamp(p, eps, 1 - eps)
    return -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()

check('BCELoss', bce_scratch(probs, target), nn.BCELoss()(probs, target))

# BCEWithLogitsLoss (numerically stable: max(x,0) - x*y + log(1+exp(-|x|)))
def bce_with_logits_scratch(x, y):
    return (torch.clamp(x, min=0) - x*y + torch.log(1 + torch.exp(-torch.abs(x)))).mean()

check('BCEWithLogits', bce_with_logits_scratch(logits, target), nn.BCEWithLogitsLoss()(logits, target))

# Equivalence: both should give the same result when input is probabilities vs logits
check('BCE equiv', nn.BCELoss()(probs, target), nn.BCEWithLogitsLoss()(logits, target), atol=1e-5)

# Show numerical instability of BCELoss on large logits
large_logits = torch.tensor([100.0, -100.0])
large_probs  = torch.sigmoid(large_logits)
large_target = torch.tensor([1.0, 0.0])
print(f'\nLarge logit test (logits=±100):')
print(f'  sigmoid(100) = {large_probs[0].item():.6f}  (should be 1.0000, log→0, NaN)')
print(f'  BCELoss result:            {nn.BCELoss()(large_probs, large_target).item()}')
print(f'  BCEWithLogitsLoss result:  {nn.BCEWithLogitsLoss()(large_logits, large_target).item():.6f}')

## Section 4 — Multi-class Classification: CrossEntropyLoss, NLLLoss

`CrossEntropyLoss = LogSoftmax + NLLLoss`. PyTorch's CrossEntropyLoss is numerically stable; never apply Softmax before it.

In [ ]:
logits = torch.tensor([[1.0, 2.0, 0.5],
                        [0.0, 1.0, 3.0],
                        [2.0, 0.0, 1.0]])
targets = torch.tensor([1, 2, 0])  # class indices
N = logits.size(0)

# NLLLoss from scratch (expects log-probabilities)
def nll_scratch(log_probs, y):
    return -log_probs[range(len(y)), y].mean()

log_probs = F.log_softmax(logits, dim=-1)
check('NLLLoss', nll_scratch(log_probs, targets), nn.NLLLoss()(log_probs, targets))

# CrossEntropyLoss = LogSoftmax + NLLLoss
def cross_entropy_scratch(x, y):
    lp = x - torch.log(torch.exp(x).sum(dim=-1, keepdim=True))  # log-softmax
    return -lp[range(len(y)), y].mean()

check('CrossEntropy', cross_entropy_scratch(logits, targets), nn.CrossEntropyLoss()(logits, targets))

# Verify the equivalence:
check('CE = NLL(LSM)', nn.NLLLoss()(F.log_softmax(logits, dim=-1), targets),
                        nn.CrossEntropyLoss()(logits, targets))
print('CrossEntropyLoss = NLLLoss(LogSoftmax(logits)) ✓')

# Label smoothing
eps, C = 0.1, 3
smooth_targets = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1)
smooth_targets = smooth_targets * (1 - eps) + eps / C
ce_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, targets)
ce_manual = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
check('Label smoothing', ce_manual, ce_smooth)

## Section 5 — Multi-label & Margin Losses

SoftMarginLoss and MultiLabelSoftMarginLoss use **{-1, +1}** labels. MultiMarginLoss is a hinge loss for multi-class.

In [ ]:
# SoftMarginLoss — binary with {-1, +1} labels
x = torch.tensor([0.5, -0.3, 1.2, -0.8])
y = torch.tensor([1.0, -1.0, 1.0, -1.0])   # {-1, +1}

def soft_margin_scratch(x, y):
    return torch.log(1 + torch.exp(-y * x)).mean()

check('SoftMarginLoss', soft_margin_scratch(x, y), nn.SoftMarginLoss()(x, y))

# MultiLabelSoftMarginLoss — per-label BCE with {-1,+1}→{0,1} conversion
x_ml = torch.randn(3, 4)
y_ml = torch.tensor([[1., -1., 1., -1.],
                      [-1., 1., 1., -1.],
                      [1., 1., -1., -1.]])

def multilabel_sm_scratch(x, y):
    t = (y + 1) / 2  # {-1,+1} → {0,1}
    return -(t * F.logsigmoid(x) + (1-t) * F.logsigmoid(-x)).mean()

check('MultiLabelSoftMargin', multilabel_sm_scratch(x_ml, y_ml), nn.MultiLabelSoftMarginLoss()(x_ml, y_ml))

# MultiMarginLoss — hinge loss: sum over j≠y of max(0, margin - x[y] + x[j])
x_mm = torch.tensor([[0.5, 1.0, 0.2],
                      [1.5, 0.3, 0.8]])
y_mm = torch.tensor([1, 0])

def multi_margin_scratch(x, y, margin=1.0, p=1):
    N, C = x.shape
    loss = 0.0
    for i in range(N):
        xi, yi = x[i], y[i].item()
        for j in range(C):
            if j != yi:
                loss += max(0, margin - xi[yi] + xi[j])**p
    return torch.tensor(loss / N)

check('MultiMarginLoss', multi_margin_scratch(x_mm, y_mm),
      nn.MultiMarginLoss()(x_mm, y_mm), atol=1e-4)

## Section 6 — Distribution & Similarity Losses

Key pitfall: **KLDivLoss input must be log-probabilities**, and use `reduction='batchmean'` to match the mathematical definition.

In [ ]:
# KLDivLoss: KL(P || Q) = Σ P * log(P/Q) = Σ P*(log P - log Q)
# PyTorch input: log Q, target: P (probabilities)
N, C = 4, 5
log_q = F.log_softmax(torch.randn(N, C), dim=-1)  # model's log-probs
p = F.softmax(torch.randn(N, C), dim=-1)           # target distribution

def kl_div_scratch(log_q, p):
    return (p * (p.log() - log_q)).sum() / N  # batchmean = sum/N

pytorch_kl = nn.KLDivLoss(reduction='batchmean')(log_q, p)
check('KLDivLoss (batchmean)', kl_div_scratch(log_q, p), pytorch_kl)

# Show the 'mean' bug: reduction='mean' divides by N*C instead of N
kl_mean = nn.KLDivLoss(reduction='mean')(log_q, p)
print(f'\nKLDivLoss reduction comparison:')
print(f'  batchmean (correct):  {pytorch_kl.item():.4f}')
print(f'  mean (wrong! /N*C):   {kl_mean.item():.4f}')
print(f'  ratio: {(pytorch_kl/kl_mean).item():.1f} ≈ C={C}')

In [ ]:
# MarginRankingLoss: max(0, -y*(x1-x2) + margin)
x1 = torch.tensor([0.8, 0.3, 1.2, -0.5])
x2 = torch.tensor([0.2, 0.9, 0.5,  0.1])
y  = torch.tensor([1., -1., 1., -1.])  # y=+1: x1 should rank higher; y=-1: x2 should

def margin_ranking_scratch(x1, x2, y, margin=0.0):
    return torch.clamp(-y * (x1 - x2) + margin, min=0).mean()

check('MarginRankingLoss', margin_ranking_scratch(x1, x2, y), nn.MarginRankingLoss()(x1, x2, y))

# CosineEmbeddingLoss: 1 - cos(x1,x2) if y=1, max(0, cos-margin) if y=-1
x1e = torch.randn(4, 8)
x2e = torch.randn(4, 8)
ye  = torch.tensor([1., -1., 1., -1.])

def cosine_embedding_scratch(x1, x2, y, margin=0.0):
    cos = F.cosine_similarity(x1, x2, dim=1)
    loss = torch.where(y == 1, 1 - cos, torch.clamp(cos - margin, min=0))
    return loss.mean()

check('CosineEmbeddingLoss', cosine_embedding_scratch(x1e, x2e, ye),
      nn.CosineEmbeddingLoss()(x1e, x2e, ye))

# HingeEmbeddingLoss: x if y=1, max(0, margin-x) if y=-1
x_h = torch.tensor([0.5, -0.2, 1.5, 0.3])
y_h = torch.tensor([1., -1., 1., -1.])

def hinge_embedding_scratch(x, y, margin=1.0):
    return torch.where(y == 1, x, torch.clamp(margin - x, min=0)).mean()

check('HingeEmbeddingLoss', hinge_embedding_scratch(x_h, y_h), nn.HingeEmbeddingLoss()(x_h, y_h))

## Section 7 — Metric Learning: TripletMarginLoss

The triplet loss pushes anchor-positive pairs together and anchor-negative pairs apart by at least a margin.

In [ ]:
torch.manual_seed(0)
B, D = 8, 16  # batch size, embedding dim
anchor   = F.normalize(torch.randn(B, D), dim=1)
positive = F.normalize(torch.randn(B, D), dim=1)
negative = F.normalize(torch.randn(B, D), dim=1)

def triplet_scratch(a, p, n, margin=1.0):
    d_ap = torch.norm(a - p, p=2, dim=1)
    d_an = torch.norm(a - n, p=2, dim=1)
    return torch.clamp(d_ap - d_an + margin, min=0).mean()

pytorch_triplet = nn.TripletMarginLoss(margin=1.0)(anchor, positive, negative)
check('TripletMarginLoss', triplet_scratch(anchor, positive, negative), pytorch_triplet)

# Triplet mining strategies
def analyse_triplets(a, p, n, margin=1.0):
    d_ap = torch.norm(a - p, p=2, dim=1)
    d_an = torch.norm(a - n, p=2, dim=1)
    easy     = d_an >= d_ap + margin          # loss=0 already
    semihard = (d_an > d_ap) & (d_an < d_ap + margin)  # positive loss
    hard     = d_an <= d_ap                   # hardest — wrong ordering
    print(f'Easy negatives:      {easy.sum().item()}/{B}')
    print(f'Semi-hard negatives: {semihard.sum().item()}/{B}')
    print(f'Hard negatives:      {hard.sum().item()}/{B}')

print('\nTriplet mining analysis:')
analyse_triplets(anchor, positive, negative)

## Section 8 — End-to-End: Siamese Network with TripletMarginLoss on MNIST

We train an embedding network to map MNIST digits into a metric space where same-class digits are close and different-class digits are far apart.

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

# Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
mnist_train = torchvision.datasets.MNIST(root='/tmp/data', train=True,  download=True, transform=transform)
mnist_test  = torchvision.datasets.MNIST(root='/tmp/data', train=False, download=True, transform=transform)

class TripletMNIST(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        self.labels  = dataset.targets
        self.label_to_indices = {c: (self.labels == c).nonzero(as_tuple=False).squeeze() for c in range(10)}
    
    def __len__(self): return len(self.dataset)
    
    def __getitem__(self, idx):
        anchor_img, anchor_label = self.dataset[idx]
        # positive: random sample from same class
        pos_indices = self.label_to_indices[anchor_label.item()]
        pos_idx = pos_indices[torch.randint(len(pos_indices), (1,))].item()
        pos_img, _ = self.dataset[pos_idx]
        # negative: random sample from different class
        neg_label = (anchor_label.item() + torch.randint(1, 10, (1,)).item()) % 10
        neg_indices = self.label_to_indices[neg_label]
        neg_idx = neg_indices[torch.randint(len(neg_indices), (1,))].item()
        neg_img, _ = self.dataset[neg_idx]
        return anchor_img, pos_img, neg_img, anchor_label

train_triplets = TripletMNIST(mnist_train)
test_triplets  = TripletMNIST(mnist_test)
train_loader = DataLoader(train_triplets, batch_size=128, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_triplets,  batch_size=256, shuffle=False, num_workers=0)
print(f'Train: {len(train_triplets)} triplets, Test: {len(test_triplets)} triplets')

In [ ]:
class EmbeddingNet(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Linear(64*7*7, 256), nn.ReLU(),
            nn.Linear(256, embed_dim)
        )
    
    def forward(self, x):
        x = self.conv(x).flatten(1)
        return F.normalize(self.fc(x), dim=1)  # L2-normalize embeddings

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmbeddingNet(embed_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
triplet_loss = nn.TripletMarginLoss(margin=0.5)
print(f'Device: {device}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
train_losses = []

for epoch in range(5):
    model.train()
    total_loss = 0
    for batch_idx, (a, p, n, _) in enumerate(train_loader):
        a, p, n = a.to(device), p.to(device), n.to(device)
        optimizer.zero_grad()
        ea, ep, en = model(a), model(p), model(n)
        loss = triplet_loss(ea, ep, en)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'  Epoch {epoch+1}, Batch {batch_idx}: loss={loss.item():.4f}')
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/5 — avg triplet loss: {avg_loss:.4f}')

In [ ]:
# Visualise the learned embedding space (project to 2D via PCA)
model.eval()
all_embeddings, all_labels = [], []

with torch.no_grad():
    for imgs, _, _, labels in DataLoader(mnist_test, batch_size=512):
        embs = model(imgs.to(device)).cpu()
        all_embeddings.append(embs)
        all_labels.append(labels)
        if len(all_embeddings) * 512 > 5000: break  # use 5k samples

embeddings = torch.cat(all_embeddings).numpy()
labels = torch.cat(all_labels).numpy()

# PCA to 2D
from numpy.linalg import svd
emb_centered = embeddings - embeddings.mean(0)
_, _, Vt = svd(emb_centered, full_matrices=False)
proj = emb_centered @ Vt[:2].T

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(proj[:, 0], proj[:, 1], c=labels, cmap='tab10', alpha=0.5, s=8)
plt.colorbar(scatter, ax=ax, label='Digit class')
ax.set_title('MNIST Embeddings (PCA to 2D) — Trained with TripletMarginLoss')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 3))
plt.plot(range(1, 6), train_losses, 'o-', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Triplet Loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()